# Unix-terminal. SSH


## 1. Подключение и удалённая команда

SSH создаёт зашифрованное соединение с удалённой машиной. Адрес имеет форму `user@host`; без пользователя берётся текущее локальное имя.

```bash
# Открыть интерактивную сессию.
ssh user@host

# Выполнить одну команду на сервере и завершить соединение.
ssh user@host whoami
ssh user@host 'uname -a'

# Подключиться к нестандартному порту.
ssh -p 2222 user@host
```

`-p` задаёт порт, `-V` показывает версию клиента. `hostname` печатает имя машины.

`ssh -G host` не подключается к серверу: он печатает итоговые параметры клиента.

Кавычки определяют, где раскроется переменная:

```bash
ssh course "echo $HOME"  # локальная оболочка
ssh course 'echo $HOME'  # удалённая оболочка
```


In [ ]:
%%bash
VALUE=local
bash -c "VALUE=remote; echo \"double quotes: $VALUE\""
bash -c 'VALUE=remote; echo "single quotes: $VALUE"'

ssh -V
which ssh
ssh -G student@server.example > /tmp/course-ssh-effective.conf 2>/dev/null
head -n 10 /tmp/course-ssh-effective.conf


### Вопрос

Локальный `$HOME` равен `/home/local`, удалённый — `/home/student`. Что выведут `ssh course "echo $HOME"` и `ssh course 'echo $HOME'`?

<details>
<summary>Ответ</summary>

Первая команда передаст уже раскрытый локальный путь `/home/local`. Вторая передаст буквальный `$HOME`, который удалённая оболочка раскроет в `/home/student`.

</details>


## 2. Ключи SSH

Пара состоит из закрытого и открытого ключа. Закрытый ключ остаётся у владельца; открытый добавляется на сервер в `~/.ssh/authorized_keys`. Парольная фраза защищает закрытый файл при краже.

На Linux закрытый ключ должен быть недоступен другим пользователям. В курсе используем `chmod 400 private_key`: чтение только владельцу. OpenSSH откажется использовать ключ с избыточно широкими правами.

`ssh-keygen` создаёт пару ключей, а также умеет показывать fingerprint и работать с файлами ключей.

- `-t` — тип ключа;
- `-f` — путь;
- `-C` — комментарий;
- `-N` — парольная фраза;
- `-q` — убрать обычные сообщения;
- `-l -f public_key` — показать fingerprint.

`ed25519` — обычный современный выбор. `rsa` нужен для совместимости. `ed25519-sk` и `ecdsa-sk` используют FIDO-устройство. DSA использовать не следует.

Тип ключа относится к аутентификации. Трафик шифруется отдельно согласованным алгоритмом, например ChaCha20-Poly1305 или AES-GCM.

Без config закрытый ключ выбирают через `ssh -i`. Опция `IdentitiesOnly=yes` запрещает клиенту перебирать другие ключи:

```bash
chmod 400 ~/.ssh/course_ed25519
ssh -o IdentitiesOnly=yes -i ~/.ssh/course_ed25519 user@host
```


In [ ]:
%%bash
rm -f /tmp/course-ssh/course_key /tmp/course-ssh/course_key.pub
ssh-keygen -q -t ed25519 -N '' -C 'course-demo' \
  -f /tmp/course-ssh/course_key
chmod 400 /tmp/course-ssh/course_key

ls -l /tmp/course-ssh/course_key*
ssh-keygen -lf /tmp/course-ssh/course_key.pub
cat /tmp/course-ssh/course_key.pub


### Вопрос

Какой файл можно передать на сервер и какие права должны быть у закрытого `course_key`?

<details>
<summary>Ответ</summary>

Передают только `course_key.pub`. Закрытый `course_key` не покидает доверенную машину; в курсе устанавливаем ему права `400`.

</details>


## 3. SSH-config


`~` обозначает домашний каталог. `~/.ssh/config` хранит параметры подключений:

- `Host` — короткое имя;
- `HostName` — адрес;
- `User` — пользователь;
- `Port` — порт;
- `IdentityFile` — закрытый ключ.

`ssh -F file` использует указанный config вместо стандартного. `ssh -G alias` показывает итоговые параметры до подключения.


In [ ]:
%%bash
mkdir -p /tmp/course-ssh
cat > /tmp/course-ssh/config <<'EOF'
Host course
    HostName server.example
    User student
    Port 2222
    IdentityFile ~/.ssh/course_ed25519
EOF

chmod 600 /tmp/course-ssh/config
ssh -F /tmp/course-ssh/config -G course \
  > /tmp/course-ssh/effective.conf 2>/dev/null
head -n 10 /tmp/course-ssh/effective.conf


### Вопрос

Как подключиться с ключом `course_ed25519` без config и какое поле укажет тот же ключ в config?

<details>
<summary>Ответ</summary>

Без config: `ssh -i ~/.ssh/course_ed25519 user@host`. В config используется `IdentityFile ~/.ssh/course_ed25519`.

</details>


## 4. Проверка сервера и диагностика


### Проверка соединения

`ssh -Tvvv user@host true` проверяет соединение и аутентификацию без интерактивного терминала: `-T` отключает псевдотерминал, `-vvv` включает максимальную клиентскую диагностику, `true` сразу завершает удалённую команду.

`-o Name=Value` передаёт одну настройку клиента. `ConnectTimeout=5` ограничивает установку соединения.


In [ ]:
%%bash
ssh -Tvvv -o ConnectTimeout=5 user@host true \
  > connection.out 2> connection-debug.txt
echo "$?" > connection-exit-code.txt


### Ключ сервера и `known_hosts`

При первом подключении SSH показывает fingerprint ключа сервера. После подтверждения ключ сохраняется в `~/.ssh/known_hosts`. Неожиданную смену ключа сначала проверяют по доверенному каналу.

`ssh-keyscan host` получает публичный ключ сервера, но не подтверждает его подлинность.

`ssh-keygen` здесь не создаёт новый ключ: `ssh-keygen -F host -f file` ищет запись хоста в выбранном `known_hosts`, `ssh-keygen -R host -f file` удаляет её.


In [ ]:
%%bash
touch /tmp/course-ssh/known_hosts
ssh-keyscan server.example \
  >> /tmp/course-ssh/known_hosts 2>/dev/null || true
ssh-keygen -F server.example \
  -f /tmp/course-ssh/known_hosts || true


### Вопрос

SSH сообщает, что ключ знакомого сервера изменился. Почему нельзя сразу удалить старую запись?

<details>
<summary>Ответ</summary>

Изменение может означать подмену сервера. Новый fingerprint сначала подтверждают по независимому доверенному каналу.

</details>


## 5. Передача файлов через `scp`

Удалённый путь записывается как `user@host:path`.

```bash
scp report.txt course:reports/
scp course:reports/report.txt returned.txt
```

Путь без начального `/` считается относительно домашнего каталога удалённого пользователя. `/var/tmp/report.txt` начинается от корня удалённой системы.

`-r` копирует каталог, `-P 2222` задаёт SSH-порт. Здесь используется заглавная `P`: строчная `-p` имеет другое значение.

После передачи туда и обратно размер сравнивают через `wc -c`. `cmp first second` ничего не печатает и возвращает `0`, если содержимое файлов совпадает; при различии возвращает ненулевой код.


In [ ]:
%%bash
HOST=course
mkdir -p /tmp/course-ssh/transfer
echo 'course report' > /tmp/course-ssh/transfer/report.txt

ssh "$HOST" 'mkdir -p reports'
scp /tmp/course-ssh/transfer/report.txt "$HOST:reports/"
scp "$HOST:reports/report.txt" /tmp/course-ssh/transfer/returned.txt

wc -c \
  /tmp/course-ssh/transfer/report.txt \
  /tmp/course-ssh/transfer/returned.txt
cmp \
  /tmp/course-ssh/transfer/report.txt \
  /tmp/course-ssh/transfer/returned.txt


### Вопрос

Куда попадут файлы из `scp file.txt course:reports/` и `scp file.txt course:/reports/`?

<details>
<summary>Ответ</summary>

Первый — в `reports` внутри домашнего каталога удалённого пользователя. Второй — в `/reports` от корня удалённой системы.

</details>


## 6. Синхронизация через `rsync`

`rsync` сравнивает источник и назначение и передаёт изменения:

```bash
rsync -av data/ course:backup/
```

`-a` сохраняет структуру и метаданные, `-v` показывает действия. `data/` означает содержимое каталога, `data` — сам каталог вместе с именем.

`--dry-run` или `-n` показывает план без изменений. `-i` подробно перечисляет изменения. `--delete` удаляет в назначении то, чего нет в источнике. `--exclude='pattern'` исключает совпавшие пути. Перед удалением всегда проверяют dry-run.


In [ ]:
%%bash
mkdir -p /tmp/course-ssh/source /tmp/course-ssh/copy
echo alpha > /tmp/course-ssh/source/a.txt
echo beta > /tmp/course-ssh/source/b.txt

rsync -avni --delete \
  --exclude='__pycache__/' \
  /tmp/course-ssh/source/ /tmp/course-ssh/copy/

rsync -avi --delete \
  --exclude='__pycache__/' \
  /tmp/course-ssh/source/ /tmp/course-ssh/copy/


### Вопрос

Чем отличаются источники `data` и `data/` в `rsync`?

<details>
<summary>Ответ</summary>

`data` копирует сам каталог с его именем, `data/` — только содержимое каталога.

</details>


## 7. `tmux`: работа после отключения


Обычная удалённая команда и процессы, привязанные к SSH-терминалу, при разрыве соединения теряют терминал и обычно завершаются. Простой запуск через `&` не гарантирует, что процесс переживёт отключение. SSH-туннель завершается вместе с клиентом SSH.

`tmux` запускает отдельную серверную сессию на удалённой машине. Она продолжает работать после отключения SSH.

```bash
# Создать интерактивную сессию.
tmux new -s work

# Отсоединиться: Ctrl+B, затем D.
# Вернуться позже.
tmux attach -t work
```

Для запуска без подключения к экрану:

```bash
tmux new-session -d -s worker 'sleep 300'
tmux list-sessions
tmux list-panes -t worker -F '#{pane_pid} #{pane_current_command}'
tmux has-session -t worker
tmux kill-session -t worker
```

`-d` создаёт detached-сессию, `-s` задаёт имя, `-t` выбирает существующую сессию, `-F` задаёт формат вывода. `#{pane_pid}` и `#{pane_current_command}` — имена полей tmux. `has-session` возвращает код `0`, если указанная сессия существует.


### Вопрос

После разрыва SSH что продолжит работать: локальный SSH-туннель или команда внутри detached-сессии `tmux`?

<details>
<summary>Ответ</summary>

Туннель завершится вместе с локальным SSH-клиентом. Detached-сессия `tmux` работает на сервере и сохранится, пока её команда или сама сессия не будут завершены.

</details>


## Дополнительно для Advanced


### Установка публичного ключа

Если вход по паролю уже работает, публичный ключ одной командой добавляется в `~/.ssh/authorized_keys`:

```bash
ssh-copy-id -i /tmp/course-ssh/course_key.pub course
```

`ssh-copy-id` подключается по SSH, создаёт нужные файлы и не добавляет уже установленный ключ повторно. Флаг `-i` выбирает публичный ключ, который надо установить.


### Туннели

В `-L 8080:127.0.0.1:8000`:

- `8080` — порт на локальной машине;
- `127.0.0.1:8000` — адрес сервиса со стороны SSH-сервера;
- `course` — сервер, через который идёт соединение.

```bash
ssh -N -L 8080:127.0.0.1:8000 course
```

`-L` создаёт локальный проброс, `-N` не запускает удалённую команду. `-o ExitOnForwardFailure=yes` завершает SSH, если порт открыть не удалось. `curl URL` выполняет HTTP-запрос и пишет тело ответа в stdout.


In [ ]:
%%bash
ssh -N -o ExitOnForwardFailure=yes \
  -L 8080:127.0.0.1:8000 course \
  > tunnel.out 2> tunnel.err &
tunnel_pid=$!

sleep 2
curl http://127.0.0.1:8080 > response.html
kill -TERM "$tunnel_pid"
wait "$tunnel_pid" 2>/dev/null || true


### Окна и вывод `tmux`

Иерархия tmux: **сессия → окна → панели**. В примере сессия называется `course-lab`, окна — `system` и `worker`; в каждом окне пока одна панель.

```bash
tmux new-session -d -s course-lab -n system 'uname -a; sleep 300'
tmux new-window -t course-lab -n worker 'uptime; sleep 300'
tmux list-windows -t course-lab
tmux capture-pane -p -t course-lab:system
```

Команды читаются так:

- `new-session -d -s course-lab -n system COMMAND` — создать detached-сессию `course-lab`, назвать первое окно `system` и запустить в нём `COMMAND`;
- `new-window -t course-lab -n worker COMMAND` — добавить в выбранную через `-t` сессию окно `worker`;
- `list-windows -t course-lab` — показать окна этой сессии;
- `capture-pane -p -t course-lab:system` — взять экран панели из окна `system` и вывести его в stdout. В адресе `session:window` двоеточие отделяет имя сессии от имени окна.

`sleep 300` оставляет окно живым после того, как `uname` или `uptime` уже напечатали результат. Без долгой команды окно сразу закроется.
